# 08 - `ml.trip_validity_fares_final`: the final fare-collection table

The second and last deliverable table: one row per fare tap (AFC
"boarding" event) for every trip already in `ml.trip_validity_final`,
regardless of whether that trip was ultimately judged valid or invalid.

**Columns**: `fare_event_id` (surrogate PK), `trip_id` (FK to
`ml.trip_validity_final`), `event_id` (the raw AFC id, kept for
traceability), `card_id`, `integration_type`, `passenger_type`,
`fare_paid`, `subsidy`, `fare_tapped_at`.

**On the PK**: `event_id` can't be the primary key - `'0'` is a
sentinel meaning "no id assigned" in the source data and is **not
unique** (223,399 of 15,381,356 November fare taps have it). Rather
than dropping those rows or picking an arbitrary one, `fare_event_id`
is a fresh `GENERATED ALWAYS AS IDENTITY` surrogate key - rebuildable
exactly like `trip_id`/`fare_id` already are elsewhere in this project
- and the raw `event_id` (duplicates and all) is kept as an ordinary,
non-unique column instead.

**On scope**: built directly from `silver.afc_boardings` joined to
`ml.trip_validity_trips` by the same natural key
(`service_date`/`vehicle_number`/`trip_opened_at`/`trip_closed_at`)
notebook 01 already used to build `ml.trip_validity_trip_fares` - not
by rejoining through `event_id`, which would be ambiguous for every
`'0'`-sentinel row. Checked directly: for November 2023, `silver.
afc_boardings` and `ml.trip_validity_trip_fares` have the exact same
row count (15,381,356) - every fare tap in scope already matches a
trip, so this notebook only ever builds rows that have one (no `NULL`
`trip_id` case to handle in practice, though the join is still a plain
`JOIN`, not something that silently drops rows some other way -
verified below).

In [1]:
import os
from pathlib import Path

import psycopg
from psycopg import sql

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

## Connect

In [3]:
from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
print("connected")

TRIP_DATE_START = "2023-11-01"
TRIP_DATE_END = "2023-12-01"

connected


## Create the table and load it

In [4]:
conn.execute("""
    DROP TABLE IF EXISTS ml.trip_validity_fares_final CASCADE;

    CREATE TABLE ml.trip_validity_fares_final (
        fare_event_id     bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        trip_id           bigint NOT NULL REFERENCES ml.trip_validity_final (trip_id),
        event_id          text NOT NULL,
        card_id           text,
        integration_type  integer,
        passenger_type    integer,
        fare_paid         double precision,
        subsidy           double precision,
        fare_tapped_at    timestamptz NOT NULL
    );
""")

conn.execute(
    """
    INSERT INTO ml.trip_validity_fares_final (
        trip_id, event_id, card_id, integration_type, passenger_type,
        fare_paid, subsidy, fare_tapped_at
    )
    SELECT
        t.trip_id,
        b.event_id,
        b.card_id,
        b.integration_type,
        b.passenger_type,
        b.fare_paid,
        b.subsidy_value,
        b.boarding_at
    FROM silver.afc_boardings b
    JOIN ml.trip_validity_trips t
      ON t.trip_date = b.service_date
     AND t.bus_id = CASE WHEN length(b.vehicle_number) < 5
                          THEN lpad(b.vehicle_number, 5, '0') ELSE b.vehicle_number END
     AND t.trip_opening_timestamp = b.trip_opened_at
     AND t.trip_closing_timestamp = b.trip_closed_at
    WHERE b.service_date >= %(start)s AND b.service_date < %(end)s;
    """,
    {"start": TRIP_DATE_START, "end": TRIP_DATE_END},
)
conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.trip_validity_fares_final;")
    print("rows inserted:", cur.fetchone()[0])

rows inserted: 15381356


## Indexes and comments

In [5]:
conn.execute("""
    CREATE INDEX trip_validity_fares_final_trip_id_idx
        ON ml.trip_validity_fares_final (trip_id);
    CREATE INDEX trip_validity_fares_final_event_id_idx
        ON ml.trip_validity_fares_final (event_id);
    CREATE INDEX trip_validity_fares_final_tapped_at_idx
        ON ml.trip_validity_fares_final (fare_tapped_at);
""")
conn.execute("ANALYZE ml.trip_validity_fares_final;")
conn.commit()


def comment_on_column(cur: psycopg.Cursor, col: str, text: str) -> None:
    """Apply a COMMENT ON COLUMN for one trip_validity_fares_final column."""
    cur.execute(
        sql.SQL("COMMENT ON COLUMN ml.trip_validity_fares_final.{} IS {};").format(
            sql.Identifier(col), sql.Literal(text)
        )
    )


COLUMN_COMMENTS = {
    "fare_event_id": (
        "Surrogate primary key, GENERATED ALWAYS AS IDENTITY - not the raw "
        "AFC event_id, which is unusable as a key (see the event_id "
        "comment below)."
    ),
    "trip_id": "ml.trip_validity_final.trip_id, as-is.",
    "event_id": (
        "silver.afc_boardings.event_id, as-is. NOT unique: '0' is a "
        "sentinel for 'no id assigned' in the source data (223,399 of "
        "15,381,356 November rows) and is shared across many rows - kept "
        "here for traceability only, never as a key."
    ),
    "card_id": "silver.afc_boardings.card_id, as-is.",
    "integration_type": (
        "silver.afc_boardings.integration_type, as-is (4 distinct codes)."
    ),
    "passenger_type": "silver.afc_boardings.passenger_type, as-is (26 distinct codes).",
    "fare_paid": "silver.afc_boardings.fare_paid, as-is.",
    "subsidy": "silver.afc_boardings.subsidy_value, as-is.",
    "fare_tapped_at": (
        "silver.afc_boardings.boarding_at, as-is (UTC) - when this fare was tapped."
    ),
}
with conn.cursor() as cur:
    for col, text in COLUMN_COMMENTS.items():
        comment_on_column(cur, col, text)
    cur.execute(
        sql.SQL("COMMENT ON TABLE ml.trip_validity_fares_final IS {};").format(
            sql.Literal(
                "The final Trip Validity fare-collection deliverable: one "
                "row per fare tap, matched to its trip in "
                "ml.trip_validity_final. See "
                "ml/trip_validity_model/notebooks/08_final_fares_table.ipynb."
            )
        )
    )
conn.commit()
print("indexes and comments applied")

indexes and comments applied


## Verification

In [6]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT count(*) FROM silver.afc_boardings
        WHERE service_date >= %(start)s AND service_date < %(end)s;
    """,
        {"start": TRIP_DATE_START, "end": TRIP_DATE_END},
    )
    EXPECTED_ROWS = cur.fetchone()[0]

    cur.execute("""
        SELECT
            count(*) AS total,
            count(*) FILTER (WHERE event_id = '0') AS sentinel_event_ids,
            count(DISTINCT trip_id) AS distinct_trips,
            count(*) FILTER (WHERE trip_id IS NULL) AS null_trip_id,
            count(*) FILTER (WHERE fare_paid < 0) AS negative_fares
        FROM ml.trip_validity_fares_final;
    """)
    cols = [c.name for c in cur.description]
    summary = dict(zip(cols, cur.fetchone(), strict=True))
    print(summary)

if summary["total"] != EXPECTED_ROWS:
    msg = (
        f"expected {EXPECTED_ROWS} rows (every November AFC boarding), "
        f"got {summary['total']}"
    )
    raise AssertionError(msg)
if summary["null_trip_id"] != 0:
    msg = "found rows with a NULL trip_id despite the NOT NULL constraint"
    raise AssertionError(msg)
if summary["negative_fares"] != 0:
    msg = "found negative fare_paid values"
    raise AssertionError(msg)

print(
    f"OK: {summary['total']} fare taps across {summary['distinct_trips']} trips, "
    f"{summary['sentinel_event_ids']} with a '0' sentinel event_id "
    f"({summary['sentinel_event_ids'] / summary['total']:.1%})"
)

{'total': 15381356, 'sentinel_event_ids': 223399, 'distinct_trips': 940988, 'null_trip_id': 0, 'negative_fares': 0}
OK: 15381356 fare taps across 940988 trips, 223399 with a '0' sentinel event_id (1.5%)
